# Notebook 4 — Product Relationship Graph (GraphFrames)
**SmartRetail: Predictive Demand Forecasting & Dynamic Pricing Engine**

### What this notebook does
1. Loads the raw `events.csv` directly (session structure lives at the event level —
   notebook 1's daily aggregation already collapsed it away)
2. Restricts to `addtocart`/`transaction` events — views are too sparse and noisy a
   co-occurrence signal; a cart/purchase basket is a much stronger "these items are
   related" signal
3. Sessionizes per visitor with a 30-minute inactivity gap, and caps basket size before
   generating pairs — this keeps pair generation bounded instead of risking a
   quadratic-blowup self-join on lifetime visitor history
4. Builds per-session item baskets and derives weighted co-occurrence pairs directly
   with DataFrame operations — this is the **load-bearing** computation; it does not
   depend on GraphFrames succeeding
5. Builds a `GraphFrame` on top of those same vertices/edges and runs **PageRank** as
   the actual graph-algorithm showcase (item "importance" in the co-purchase network)
6. Exports `models/relationships.json` — top-N related items per SKU with edge weights,
   so Phase 2 can scale a price bump on a related item proportionally to how strongly
   it co-occurs with the surging item

---
### Before you run
Uses the same Drive layout as notebooks 1–3, reading raw data instead of processed:
```
MyDrive/
└── SmartRetail/
    ├── data/
    │   └── raw/
    │       └── events.csv
    └── models/
        └── relationships.json   ← written by this notebook
```

### If GraphFrames fails to load
GraphFrames ships a separate JVM package per Spark/Scala build, and neither of us can
verify the exact coordinate from outside Colab. If `SparkSession.builder.getOrCreate()`
in Step 4 hangs/errors on package resolution, or a `GraphFrame` call throws
`NoSuchMethodError`, look up the correct build for PySpark 3.5.x at
[spark-packages.org/package/graphframes/graphframes](https://spark-packages.org/package/graphframes/graphframes)
and update `GRAPHFRAMES_PACKAGE` in Step 3 — no other cell needs to change. If the
**Python** `graphframes` import itself fails (rather than the jar), that's a pip-wrapper
version mismatch, not a coordinate problem.

Steps 1–8 and 11 (everything up to and including `relationships.json`) do not depend on
GraphFrames resolving — only Steps 9–10 (the `GraphFrame`/PageRank showcase) do. If
GraphFrames can't be made to work in your Colab session, skip 9–10 and the notebook's
actual deliverable is unaffected.


## Step 1 — Install PySpark & GraphFrames

In [ ]:
!pip install pyspark==3.5.1 -q
!pip install graphframes -q
print("PySpark + GraphFrames (Python wrapper) installed.")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Configuration
**Only edit this cell.** All paths, the GraphFrames package coordinate, and the
sessionization/graph parameters below derive from here.

In [ ]:
# ── Edit these two lines to match your Drive layout ────────────────────────────
RAW_DATA_PATH = "/content/drive/MyDrive/SmartRetail/data/raw/"
MODELS_PATH   = "/content/drive/MyDrive/SmartRetail/models/"
# ───────────────────────────────────────────────────────────────────────────────

EVENTS_CSV          = RAW_DATA_PATH + "events.csv"
RELATIONSHIPS_JSON  = MODELS_PATH + "relationships.json"

# See "If GraphFrames fails to load" above if this coordinate doesn't resolve.
GRAPHFRAMES_PACKAGE = "graphframes:graphframes:0.8.3-spark3.5-s_2.12"

SESSION_GAP_SECONDS = 1800   # 30 minutes of visitor inactivity starts a new session
MAX_SESSION_ITEMS   = 20     # cap basket size before generating pairs (bounds pair-explosion)
BASKET_EVENT_TYPES  = ["addtocart", "transaction"]
TOP_N_RELATED        = 5     # related items kept per SKU in relationships.json

print("Config ready.")
print(f"  Events in         : {EVENTS_CSV}")
print(f"  Relationships out : {RELATIONSHIPS_JSON}")
print(f"  GraphFrames pkg   : {GRAPHFRAMES_PACKAGE}")
print(f"  Session gap       : {SESSION_GAP_SECONDS}s")
print(f"  Max basket size   : {MAX_SESSION_ITEMS}")
print(f"  Basket events     : {BASKET_EVENT_TYPES}")

## Step 4 — Initialise SparkSession (with GraphFrames)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SmartRetail-GraphFrames")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.jars.packages", GRAPHFRAMES_PACKAGE)
    .config("spark.jars.repositories", "https://repos.spark-packages.org")
    .config("spark.sql.checkpoint.dir", "/content/spark-checkpoints")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setCheckpointDir("/content/spark-checkpoints")   # GraphFrames' pregel-based algorithms (e.g. connected components) need this; PageRank does not, but it's harmless to set

print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

## Step 5 — Load Raw Events
Session structure lives at the individual event level (`visitorid`, `timestamp`,
`itemid`, `event`) — notebook 1's daily aggregation already collapsed this away, so we
read the raw CSV directly rather than notebook 1's output.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(EVENTS_CSV)
)

events = events_raw.withColumn("event_ts", F.to_timestamp(F.col("timestamp") / 1000))

print(f"Raw events: {events.count():,}")
events.groupBy("event").count().orderBy(F.col("count").desc()).show()

## Step 6 — Restrict to Basket Events & Sessionize
Views are the noisiest, least intentional signal of "these products relate to each
other" — a browsing session can wander across unrelated items. Add-to-cart and
transaction events are far sparser and a much stronger co-occurrence signal, so we
restrict to those before building sessions.

A session is a run of a visitor's events with no gap longer than
`SESSION_GAP_SECONDS` — a standard clickstream sessionization heuristic. This also
keeps basket sizes small by construction, which is what makes Step 8's pair generation
safe (no quadratic blowup from a visitor's entire lifetime history).

In [ ]:
basket_events = events.filter(F.col("event").isin(BASKET_EVENT_TYPES))

visitor_window = Window.partitionBy("visitorid").orderBy("event_ts")

sessionized = (
    basket_events
    .withColumn("prev_ts", F.lag("event_ts").over(visitor_window))
    .withColumn("gap_seconds", F.col("event_ts").cast("long") - F.col("prev_ts").cast("long"))
    .withColumn(
        "new_session_flag",
        F.when(F.col("prev_ts").isNull(), 1)
         .when(F.col("gap_seconds") > SESSION_GAP_SECONDS, 1)
         .otherwise(0)
    )
    .withColumn("session_num", F.sum("new_session_flag").over(visitor_window))
    .withColumn("session_id", F.concat_ws("_", F.col("visitorid"), F.col("session_num")))
)

print(f"Basket events (addtocart/transaction): {basket_events.count():,}")
print(f"Distinct sessions: {sessionized.select('session_id').distinct().count():,}")
sessionized.select("visitorid", "event_ts", "itemid", "event", "gap_seconds", "session_id") \
    .orderBy("visitorid", "event_ts").show(10)

## Step 7 — Build Per-Session Item Baskets
One row per session: the distinct set of items involved in it. Sessions with only one
item can't produce a co-occurrence pair and are dropped; sessions above
`MAX_SESSION_ITEMS` are dropped too — almost certainly bot/scraper activity, not a
genuine shopping basket, and exactly the pathological case that would blow up pair
generation.

In [ ]:
session_items = (
    sessionized
    .groupBy("session_id")
    .agg(F.array_distinct(F.collect_list("itemid")).alias("items"))
    .withColumn("basket_size", F.size("items"))
    .filter((F.col("basket_size") >= 2) & (F.col("basket_size") <= MAX_SESSION_ITEMS))
)

print(f"Sessions with 2-{MAX_SESSION_ITEMS} distinct basket items: {session_items.count():,}")
session_items.select("basket_size").describe().show()

## Step 8 — Generate Co-Occurrence Pairs & Edge Weights
Each session's basket is already capped at `MAX_SESSION_ITEMS`, so a self-join **on
`session_id`** is safe here — unlike a lifetime-history self-join on `visitorid`, each
join key now only matches within one small, capped basket, so there's no quadratic
blowup. `item1 < item2` keeps one direction per pair and drops self-pairs. The weight
is how many distinct baskets each pair co-occurred in.

In [ ]:
exploded = session_items.select("session_id", F.explode("items").alias("item"))

pairs = (
    exploded.alias("a")
    .join(exploded.alias("b"), on="session_id")
    .filter(F.col("a.item") < F.col("b.item"))
    .select(F.col("a.item").alias("item1"), F.col("b.item").alias("item2"))
)

edge_weights = (
    pairs
    .groupBy("item1", "item2")
    .agg(F.count("*").alias("weight"))
    .cache()
)

print(f"Unique co-occurring item pairs: {edge_weights.count():,}")
edge_weights.orderBy(F.col("weight").desc()).show(10)

## Step 9 — Build Vertices & Edges for GraphFrame
Relationships are symmetric — if A relates to B, B relates to A — so each pair becomes
two directed edges. This is also the point where GraphFrames actually enters the
notebook; everything above is plain DataFrame work and already fully determines
`relationships.json` (Step 11).

In [ ]:
from graphframes import GraphFrame

vertices = events.select(F.col("itemid").alias("id")).distinct()

edges = (
    edge_weights.select(F.col("item1").alias("src"), F.col("item2").alias("dst"), "weight")
    .union(edge_weights.select(F.col("item2").alias("src"), F.col("item1").alias("dst"), "weight"))
    .cache()
)

g = GraphFrame(vertices, edges)
print(f"Vertices: {g.vertices.count():,}")
print(f"Edges (bidirectional): {g.edges.count():,}")

## Step 10 — Run PageRank
The graph-algorithm showcase: PageRank surfaces items that are structurally "central"
to co-purchase behavior — items that relate to many other frequently-co-purchased
items, not just items with the single largest basket count.

In [ ]:
pagerank_result = g.pageRank(resetProbability=0.15, maxIter=10)

print("Top 10 items by PageRank (most central to co-purchase behavior):")
pagerank_result.vertices.orderBy(F.col("pagerank").desc()).show(10)

## Step 11 — Export `relationships.json`
Derived directly from `edges` (Step 9's DataFrame, not the `GraphFrame` object), so
this step runs regardless of whether Steps 9–10 succeeded. Each SKU keeps its top
`TOP_N_RELATED` related items **with the co-occurrence weight**, so Phase 2 can scale a
related item's price bump proportionally to relationship strength rather than applying
a flat adjustment to every related item alike.

JSON object keys are strings even though `itemid` is an integer — Phase 2's lookup code
must `str()` the incoming id before indexing into this file.

In [ ]:
w = Window.partitionBy("src").orderBy(F.col("weight").desc())

top_related = (
    edges
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") <= TOP_N_RELATED)
    .orderBy("src", "rank")
)

related_rows = top_related.select("src", "dst", "weight").collect()

relationships = {}
for row in related_rows:
    relationships.setdefault(str(row["src"]), []).append(
        {"related": int(row["dst"]), "weight": int(row["weight"])}
    )

print(f"Items with at least one related product: {len(relationships):,}")

import os
import json

os.makedirs(MODELS_PATH, exist_ok=True)
with open(RELATIONSHIPS_JSON, "w") as f:
    json.dump(relationships, f, indent=2)

print(f"Relationships written to: {RELATIONSHIPS_JSON}")

sample_key = next(iter(relationships))
print(f"\nSample entry (itemid {sample_key}):")
print(json.dumps({sample_key: relationships[sample_key]}, indent=2))

## Step 12 — Final Summary

In [ ]:
print("═" * 55)
print("  NOTEBOOK 4 — PRODUCT RELATIONSHIP GRAPH COMPLETE")
print("═" * 55)
print()
print(f"Items with related products : {len(relationships):,}")
print(f"Relationships written to    : {RELATIONSHIPS_JSON}")
print()
print("What Phase 2 (consumer_engine.py) expects:")
print("  Load relationships.json as a dict keyed by str(itemid).")
print("  On a surge for item X, look up relationships[str(X)] -> list of {related, weight}.")
print("  Scale each related item's price bump by its weight relative to X's own bump.")

spark.stop()
print()
print("SparkSession closed.")